In [15]:
import pandas as pd
import os
import shutil
import matplotlib.pyplot as plt
import os
import glob
import torch
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np
import time
from sklearn.utils import resample

In [1]:
BASE_DIR = "" 
BASE_IMAGE_DIR = ""
CSV_FILE_NAME = "montreal_circuit-labels.csv"
EXPORT_CSV_FILE_NAME = "data.csv"

SyntaxError: incomplete input (840475559.py, line 2)

In [17]:
# load data
df = pd.read_csv(os.path.join(BASE_DIR,CSV_FILE_NAME))

In [18]:
df.describe()

,v,w
count,1411.000000,1411.000000
mean,4.321916,-0.002835
std,0.715543,0.626324
min,4.000000,-1.879130
25%,4.000000,-0.311910
50%,4.000000,-0.001200
75%,4.000000,0.121105
max,6.840000,2.279820


In [19]:
new_df = df[df['image'].apply(lambda img: os.path.isfile(os.path.join(BASE_IMAGE_DIR, img)))]

In [20]:
# update image index
# Add 'image/' prefix to each row in the 'image' column
new_df.loc[:,'image'] = new_df['image'].apply(lambda x: f'images/{x}')

# save 
new_df.to_csv(os.path.join(BASE_DIR,EXPORT_CSV_FILE_NAME),index=False)
new_df.describe()

,v,w
count,1411.000000,1411.000000
mean,4.321916,-0.002835
std,0.715543,0.626324
min,4.000000,-1.879130
25%,4.000000,-0.311910
50%,4.000000,-0.001200
75%,4.000000,0.121105
max,6.840000,2.279820


In [21]:
# load new 

# image proccess

In [22]:
df = pd.read_csv(os.path.join(BASE_DIR,EXPORT_CSV_FILE_NAME))

In [23]:
# start time
start_time = time.time()
error_images = 0
complete_images = 0
IMG_HEIGHT = 240
proccessed_image_dir = os.path.join(BASE_DIR,'cropped_images')

# make dir for processed images
os.makedirs(proccessed_image_dir,exist_ok=True)


for image_name in df['image']:
    # 1. check image already proccessed or not
    full_path = os.path.join(os.path.join(proccessed_image_dir, image_name))
    if os.path.exists(full_path):
        continue

    # if proccessed is not exist on the images folder
    # GET THE FULL IMAGE INPUT PATH IF CURRENT `circuit`
    image_path = os.path.join(BASE_DIR,image_name)

    # Transform to tensor
    to_tensor = transforms.ToTensor()  # Converts PIL to [C,H,W] FloatTensor in [0,1]
    to_pil = transforms.ToPILImage()   # For visualization or debug


    try:
        # Load image with PIL and convert to RGB
        img = Image.open(image_path).convert('RGB')

        # To tensor and move to GPU
        tensor = to_tensor(img).unsqueeze(0).cuda()  # Shape: [1, 3, H, W]

        # Crop last 240 rows
        cropped_tensor = tensor[:, :, -IMG_HEIGHT:, :]  # Shape: [1, 3, 240, W]

        # Move to CPU and convert to numpy image (H, W, C) in uint8
        cropped_img_np = (cropped_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)

        # Save using OpenCV
        filename = os.path.basename(image_path)
        save_path = os.path.join(proccessed_image_dir,filename)
            
        cv2.imwrite(save_path, cv2.cvtColor(cropped_img_np, cv2.COLOR_RGB2BGR))

        complete_images += 1
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        error_images += 1
        
        
        
# end time
end_time = time.time()
elapsed = end_time - start_time

print("Total images {len(df['image'])}")
print(f"Complete to process {complete_images} images.")
print(f"Failed to process {error_images} images.")
print(f"Elapsed time: {elapsed:.2f} seconds")

Total images {len(df['image'])}
Complete to process 1411 images.
Failed to process 0 images.
Elapsed time: 11.60 seconds


# Shuffle data 

In [24]:
#load dataset
df = pd.read_csv("balanced_train.csv")

In [25]:
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [26]:
df_shuffled.to_csv("balanced_train.csv", index=False)
